# Financial Report Assistant — Minh hoạ Workflow End-to-End

Notebook này chạy thật các hàm trong `src/` theo đúng 4 luồng của hệ thống hiện tại: **Qdrant + hybrid search** cho truy xuất và **agent LangGraph** cho hỏi đáp. Không mock: embedding thật, Qdrant thật, Gemini thật.

| # | Luồng | Kích hoạt bởi | Mục đích |
|---|---|---|---|
| A | **Index một filing** | `python main.py --ticker X --filing Y`, form "Index a new filing" trên sidebar, hoặc tool `index_filing` của agent | Parse HTML → chunk cha/con → embed dense + BM25 → Qdrant (chunk con) + SQLite (chunk cha) |
| B | **Truy vấn** | Mọi lần tìm kiếm | Hybrid search (dense + BM25, gộp bằng RRF) trên chunk con, có pre-filter metadata, trả về chunk cha |
| C | **Agent** | Ô chat trên Streamlit | Planner ⇄ executor → synthesizer; chat nhiều lượt; hỏi người dùng trước khi index |
| D | **Đánh giá** | `python -m src.evaluation.metrics`, `python -m src.evaluation.agent_eval` | Đo retrieval (golden set một bước) và agent (câu hỏi nhiều bước) |

## Quy ước trình bày

Mỗi hàm/lớp chính được giới thiệu theo 4 phần: **1. Tên + Input**, **2. Output**, **3. Chức năng**, **4. Tech stack**, rồi đến cell chạy thật.

## An toàn khi chạy

- **Phần A** index vào một **store tạm trong bộ nhớ** và ghi Markdown ra thư mục tạm: không đụng tới `data/qdrant`, `data/parents.sqlite3`, `data/processed` hay `manifest.json`.
- **Phần B–D** chỉ **đọc** store thật. Qdrant local mode khoá thư mục `data/qdrant`, nên phải tắt app Streamlit trước khi chạy notebook.
- **Phần C** gọi Gemini thật (khoảng 10–15 lời gọi, rải trên chuỗi fallback model). Đề nghị index ở C.6 bị **từ chối**, nên store không đổi.

In [1]:
import json, logging, os, sys, tempfile, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

def quiet_logs():
    # Mỗi module trong src/ có logger INFO riêng; hạ xuống WARNING cho output gọn.
    for name in list(logging.root.manager.loggerDict):
        if name.startswith("src"):
            logging.getLogger(name).setLevel(logging.WARNING)

from src.embeddings.embedder import get_embedding_model, get_sparse_embedding_model

embedding = get_embedding_model()          # BAAI/bge-m3, CPU
sparse = get_sparse_embedding_model()      # Qdrant/bm25 (fastembed)
quiet_logs()
print("Repo:", ROOT)

2026-10-11 03:35:31 | INFO     | src.embeddings.embedder | Loading embedding model: BAAI/bge-m3 (device=cpu)


2026-10-11 03:35:40 | INFO     | src.embeddings.embedder | Loading sparse model: Qdrant/bm25


Repo: C:\Python\Personal Project\Financial Report Assistant


# PHẦN A — Index một filing (vào store tạm)

**Các bên tham gia:** `sec_loader` (tải filing; bỏ qua ở đây vì file đã có sẵn), `filing_facts` (đọc kỳ tài chính), `parse_html` (HTML → Markdown có cấu trúc), `chunking` (chunk cha/con), `embedder` (dense + BM25), `FilingStore` + `ParentStore` (Qdrant + SQLite), `pipeline.index_filing` (ghép tất cả).

**File minh hoạ:** `data/raw/NVDA_10-Q_2026-08-26.html` — một 10-Q đã tải nhưng **không** có trong index thật, nên đây đúng là "index một filing mới".

## A.1 — `extract_filing_facts(raw_html_path) -> dict`

1. **Input:** đường dẫn file HTML thô của filing.
2. **Output:** `{"fiscal_year": int, "fiscal_period": "FY" | "Q1".."Q3", "period_end_date": "YYYY-MM-DD"}` (giá trị `None` nếu filing không khai báo).
3. **Chức năng:** đọc kỳ tài chính từ chính trang bìa inline-XBRL của filing (thẻ `dei:DocumentFiscalYearFocus`, `dei:DocumentFiscalPeriodFocus`, `dei:DocumentPeriodEndDate`), không gọi mạng. Lý do: năm của ngày nộp thường lệch năm tài chính (10-Q nộp tháng 8/2026 của NVIDIA thuộc **năm tài chính 2027**).
4. **Tech stack:** BeautifulSoup + `SoupStrainer` chỉ parse các thẻ `ix:nonNumeric`/`ix:nonFraction` cần thiết.

In [2]:
from src.preprocessing.filing_facts import extract_filing_facts

RAW = ROOT / "data" / "raw" / "NVDA_10-Q_2026-08-26.html"
facts = extract_filing_facts(RAW)
facts

{'fiscal_year': 2027, 'fiscal_period': 'Q2', 'period_end_date': '2026-07-26'}

## A.2 — `parse_html(raw_html_path) -> Path`

1. **Input:** đường dẫn HTML thô; loại form (10-K/10-Q) suy ra từ tên file.
2. **Output:** đường dẫn file Markdown vừa ghi.
3. **Chức năng:** HTML → Markdown có cấu trúc. Mỗi SEC Item thành một mục **H2** (10-Q có thêm "Part I/II —"), tiêu đề phụ in đậm thành **H3**, bảng HTML thành bảng Markdown. Bỏ "đồ đạc trang": header lặp lại, số trang, mục lục. Chữ được ghép đúng như trình duyệt hiển thị (sửa "RIS K FACTORS", "$ 2.4").
4. **Tech stack:** BeautifulSoup/lxml; chiến lược nhận diện mục theo từng loại form (`FilingProfile`).

Ở đây hàm ghi ra thư mục tạm (trỏ `processed_path` của module sang `TMP`) thay vì `data/processed`.

In [3]:
import src.preprocessing.parse_html as parse_html_module

TMP = Path(tempfile.mkdtemp(prefix="fra_notebook_"))
parse_html_module.processed_path = lambda doc_id: TMP / f"{doc_id}.md"

md_path = parse_html_module.parse_html(RAW)
markdown = md_path.read_text(encoding="utf-8")
h2 = [line[3:] for line in markdown.splitlines() if line.startswith("## ")]
h3 = [line for line in markdown.splitlines() if line.startswith("### ")]
print(f"{md_path.name}: {len(markdown):,} ký tự, {len(h2)} mục H2, {len(h3)} tiêu đề phụ H3\n")
print(*h2, sep="\n")

2026-10-11 03:35:40 | INFO     | src.preprocessing.parse_html | Parsing C:\Python\Personal Project\Financial Report Assistant\data\raw\NVDA_10-Q_2026-08-26.html (profile=10-Q)


2026-10-11 03:35:41 | INFO     | src.preprocessing.parse_html | Structural heading detection not confident; falling back to regex.


2026-10-11 03:35:41 | INFO     | src.preprocessing.parse_html | Wrote C:\Users\User\AppData\Local\Temp\fra_notebook_93jb0w68\NVDA_10-Q_2026-08-26.md (9 confirmed sections)


NVDA_10-Q_2026-08-26.md: 162,617 ký tự, 10 mục H2, 101 tiêu đề phụ H3

Preamble
Part I — Item 1. Financial Statements (Unaudited)
Part I — Item 2. Management’s Discussion and Analysis of Financial Condition and Results of Operations
Part I — Item 3. Quantitative and Qualitative Disclosures About Market Risk
Part I — Item 4. Controls and Procedures
Part II — Item 1. Legal Proceedings
Part II — Item 1A. Risk Factors
Part II — Item 2. Unregistered Sales of Equity Securities and Use of Proceeds
Part II — Item 5. Other Information
Part II — Item 6. Exhibits


## A.3 — `chunk_markdown(md_path, filing_meta) -> ChunkedFiling`

1. **Input:** file Markdown; `filing_meta` gồm `doc_id`, `ticker`, `filing_type`, `filing_date`, `filing_name`, `source`, `url` và các fiscal facts ở A.1.
2. **Output:** `ChunkedFiling(parents, children)` — hai danh sách `Document`.
3. **Chức năng — parent-child chunking:**
   - **Chunk cha** = một tiểu mục H3 (gộp nếu < 400 ký tự, tách nếu > 6.000). Đây là thứ LLM đọc.
   - **Chunk con** = cửa sổ 800 ký tự chồng lấp 120 cho văn bản; bảng giữ nguyên (bảng dài cắt theo hàng, lặp header, kèm caption). Đây là thứ được embed và tìm kiếm.
   - Mỗi chunk mang metadata để lọc trước: `item_code`, `part`, `section`, `subsection`, `content_type`, `fiscal_year`, `fiscal_period`, `parent_id`…, và một dòng header "Company/Filing/Section" ở đầu nội dung.
4. **Tech stack:** `RecursiveCharacterTextSplitter` của `langchain-text-splitters`; `Document` của `langchain-core`.

In [4]:
from src.preprocessing.chunking import chunk_markdown
from src.utils.helpers import parse_doc_id

doc_id = RAW.stem
filing_meta = {
    **parse_doc_id(doc_id), "doc_id": doc_id, "filing_name": "NVIDIA CORP 10-Q 2026-08-26",
    "source": RAW.name, "url": "(minh hoạ)", **facts,
}
chunked = chunk_markdown(md_path, filing_meta)
print(f"{len(chunked.parents)} chunk cha, {len(chunked.children)} chunk con\n")

parent = max((p for p in chunked.parents if "Revenue" in (p.metadata.get("subsection") or "")), key=lambda p: len(p.page_content))
kids = [c for c in chunked.children if c.metadata["parent_id"] == parent.metadata["parent_id"]]
print({k: parent.metadata[k] for k in ("section", "subsection", "item_code", "part", "fiscal_year", "fiscal_period", "parent_id")})
print("\n" + parent.page_content[:700])
print(f"\n{len(kids)} chunk con:", [(c.metadata["chunk_id"], c.metadata["content_type"], len(c.page_content)) for c in kids])

2026-10-11 03:35:41 | INFO     | src.preprocessing.chunking | Chunking C:\Users\User\AppData\Local\Temp\fra_notebook_93jb0w68\NVDA_10-Q_2026-08-26.md


2026-10-11 03:35:41 | INFO     | src.preprocessing.chunking | Produced 80 parents and 281 children across 10 sections for doc_id=NVDA_10-Q_2026-08-26


80 chunk cha, 281 chunk con

{'section': 'Part I — Item 2. Management’s Discussion and Analysis of Financial Condition and Results of Operations', 'subsection': 'Concentration of Revenue', 'item_code': '2', 'part': 'I', 'fiscal_year': 2027, 'fiscal_period': 'Q2', 'parent_id': 49}

Company: NVIDIA CORP 10-Q 2026-08-26 (NVDA)
Filing: 10-Q filed 2026-08-26, fiscal 2027 Q2
Section: Part I — Item 2. Management’s Discussion and Analysis of Financial Condition and Results of Operations
Subsection: Concentration of Revenue

### Concentration of Revenue

We refer to customers who purchase products directly from NVIDIA as direct customers, such as AIBs, distributors, ODMs, OEMs, CSPs, AI model makers, and system integrators. Certain direct customers may use either internal resources or third-party system integrators to complete their build. We refer to indirect customers as those who purchase products through our direct customers; indirect customers include CSPs, AI clouds, AI 

4 chunk con: [(1

## A.4 — Embedding: `get_embedding_model()` và `get_sparse_embedding_model()`

1. **Input:** không có; đọc tên model và thiết bị từ `src/utils/config.py`.
2. **Output:** model dense (`HuggingFaceEmbeddings`, BAAI/bge-m3, 1024 chiều, chuẩn hoá) và model sparse (`SparseTextEmbedding` "Qdrant/bm25").
3. **Chức năng:** mỗi chunk con có **hai vector**. Dense bắt nghĩa (đa ngôn ngữ); BM25 bắt đúng từ khoá ("Part II Item 1", "NVLink") mà dense hay bỏ lỡ. Qdrant áp IDF cho vector sparse.
4. **Tech stack:** sentence-transformers qua `langchain-huggingface`; `fastembed`. Chạy trên CPU.

In [5]:
dense_vector = embedding.embed_query("data center revenue")
sparse_vector = next(iter(sparse.query_embed("data center revenue")))
print(len(dense_vector), "chiều dense")
print(len(sparse_vector.indices), "token BM25 (chỉ số, trọng số):", list(zip(sparse_vector.indices.tolist(), sparse_vector.values.round(3).tolist())))

1024

 chiều dense
3 token BM25 (chỉ số, trọng số): [(1714869808, 1), (989116115, 1), (302067699, 1)]


## A.5 — `FilingStore(client, embedding, sparse_embedding, parents)` và `.add_filing(parents, children)`

1. **Input:** một `QdrantClient`, hai model embedding, một `ParentStore` (SQLite).
2. **Output:** `add_filing` không trả gì; chunk cha vào SQLite, chunk con vào Qdrant.
3. **Chức năng:** một collection Qdrant chung cho mọi filing. Mỗi chunk con là một point có 2 vector (`dense`, `bm25`) và payload phẳng (metadata + nội dung). Point id = `uuid5(doc_id:chunk_id)` nên index lại cùng chunk sẽ ghi đè chứ không nhân đôi.
4. **Tech stack:** `qdrant-client` (ở đây chạy `:memory:`; app dùng local mode `data/qdrant` hoặc server qua `QDRANT_URL`), SQLite.

Để chạy nhanh trên CPU, chỉ embed các chunk con thuộc **Part I — Item 2 (MD&A)**.

In [6]:
from qdrant_client import QdrantClient
from src.retrieval.parent_store import ParentStore
from src.retrieval.vector_store import FilingStore

demo_store = FilingStore(QdrantClient(":memory:"), embedding, sparse, ParentStore(TMP / "parents.sqlite3"), collection_name="notebook_demo")
mdna = [c for c in chunked.children if c.metadata.get("part") == "I" and c.metadata.get("item_code") == "2"]

start = time.perf_counter()
demo_store.add_filing(chunked.parents, mdna)
print(f"Đã embed {len(mdna)} chunk con trong {time.perf_counter() - start:.1f}s -> {demo_store.count()} point trong Qdrant tạm")
for hit in demo_store.search("data center revenue", k=3, mode="hybrid"):
    print("-", hit.metadata["subsection"], "|", " ".join(hit.page_content.split())[:150])

2026-10-11 03:35:41 | INFO     | src.retrieval.parent_store | Stored 80 parents


2026-10-11 03:36:26 | INFO     | src.retrieval.vector_store | Created collection 'notebook_demo' (dense size=1024, sparse=bm25)


2026-10-11 03:36:26 | INFO     | src.retrieval.vector_store | Upserted 64/70 children


2026-10-11 03:36:28 | INFO     | src.retrieval.vector_store | Upserted 70/70 children


Đã embed 70 chunk con trong 46.8s -> 70 point trong Qdrant tạm
- Second Quarter of Fiscal Year 2027 Summary | Company: NVIDIA CORP 10-Q 2026-08-26 (NVDA) Filing: 10-Q filed 2026-08-26, fiscal 2027 Q2 Section: Part I — Item 2. Management’s Discussion and Analys
- Recent Developments, Future Objectives and Challenges | Company: NVIDIA CORP 10-Q 2026-08-26 (NVDA) Filing: 10-Q filed 2026-08-26, fiscal 2027 Q2 Section: Part I — Item 2. Management’s Discussion and Analys
- Recent Developments, Future Objectives and Challenges | Company: NVIDIA CORP 10-Q 2026-08-26 (NVDA) Filing: 10-Q filed 2026-08-26, fiscal 2027 Q2 Section: Part I — Item 2. Management’s Discussion and Analys


## A.6 — `index_filing(ticker, filing_type, filing_date=None, *, store=None, progress=None) -> dict` *(không chạy ở đây)*

1. **Input:** ticker, loại form, ngày nộp (bỏ trống = bản mới nhất); `store` đang mở (app truyền store đã cache vì Qdrant local mode chỉ cho một client); `progress` là callback báo từng giai đoạn.
2. **Output:** `{"doc_id", "status": "indexed" | "skipped", "chunk_count"}`.
3. **Chức năng:** đường index **duy nhất** của project — CLI `main.py`, form trên sidebar và tool `index_filing` của agent đều gọi hàm này. Nó ghép đúng các bước A.1–A.5: tải từ SEC EDGAR → đọc fiscal facts → parse → chunk → embed + lưu → ghi `manifest.json`. Filing đã index thì bỏ qua (khoá là `doc_id`). Callback `progress` là thứ agent stream lên giao diện ("Fetching…", "Parsing and chunking…", "Embedding N chunks…").
4. **Tech stack:** các module ở trên; `requests` cho SEC EDGAR.

Không chạy trong notebook vì nó ghi vào store thật và `manifest.json`.

# PHẦN B — Truy vấn (store thật, chỉ đọc)

**Các bên tham gia:** `open_store` (mở store dùng chung), `FilingStore.search` (tìm chunk con), `Retriever` (lọc trước + gom chunk con thành chunk cha).

## B.1 — `open_store(embedding, sparse_embedding) -> FilingStore` và `.filing_summaries()`

1. **Input:** hai model embedding.
2. **Output:** `FilingStore` trên `data/qdrant` (local mode) hoặc Qdrant server nếu đặt `QDRANT_URL`; chunk cha ở `data/parents.sqlite3`.
3. **Chức năng:** `filing_summaries()` trả về danh mục filing đã index, gồm kỳ tài chính. Agent dùng danh mục này để chọn filter đúng giá trị thật.
4. **Tech stack:** Qdrant facet theo `doc_id`, `pandas` để hiển thị.

In [7]:
import pandas as pd
from src.retrieval.vector_store import open_store

store = open_store(embedding, sparse)
pd.DataFrame(store.filing_summaries())

2026-10-11 03:36:29 | INFO     | src.retrieval.vector_store | Opened local Qdrant store at C:\Python\Personal Project\Financial Report Assistant\data\qdrant


,doc_id,ticker,filing_type,filing_date,filing_name,fiscal_year,fiscal_period,period_end_date,chunk_count
0,GOOGL_10-K_2026-02-05,GOOGL,10-K,2026-02-05,Alphabet Inc. 10-K 2026,2025,FY,2025-12-31,647
1,MSFT_10-K_2026-07-29,MSFT,10-K,2026-07-29,MICROSOFT CORP 10-K 2026,2026,FY,2026-06-30,630
2,NVDA_10-Q_2026-05-20,NVDA,10-Q,2026-05-20,NVIDIA CORP 10-Q 2026,2027,Q1,2026-04-26,263
3,NVDA_10-K_2026-02-25,NVDA,10-K,2026-02-25,NVIDIA CORP 10-K 2026,2026,FY,2026-01-25,622


## B.2 — `FilingStore.search(query, k, query_filter=None, mode="hybrid") -> list[Document]`

1. **Input:** câu truy vấn; số kết quả `k`; filter Qdrant (áp **trong lúc** tìm, không lọc sau); `mode` là `"dense"`, `"sparse"` hoặc `"hybrid"`.
2. **Output:** các chunk con, xếp theo độ liên quan.
3. **Chức năng:** `hybrid` chạy cả hai nhánh (mỗi nhánh 50 ứng viên, cùng filter) rồi gộp bằng **Reciprocal Rank Fusion** (k = 60) trong một lệnh `query_points`. Trên golden set 35 câu: dense MRR 0,810, BM25 0,623, hybrid **0,860**.
4. **Tech stack:** Query API của Qdrant (`prefetch` + `RrfQuery`).

Câu hỏi dưới đây (lấy từ golden set) cần khớp đúng từ ngữ: "Part II … Item 1" là mục *Legal Proceedings* của 10-Q, mà chunk của mục này chỉ là một câu dẫn sang Part I. Cell in hạng của mục đúng trong top 20 chunk con của từng chế độ:

- **dense** không thấy nó: nội dung chunk nói về Part I chứ không về tranh chấp pháp lý;
- **BM25** xếp nó hạng 3 nhờ khớp đúng "Part II", "Item 1";
- **hybrid** đưa nó vào hạng 4 — điều dense một mình không làm được.

RRF cộng điểm theo **hạng** ở hai nhánh, nên một chunk chỉ một nhánh tìm thấy có thể bị các chunk xuất hiện ở cả hai nhánh vượt qua: với câu ngắn hơn "What is covered in Part II Item 1?", hybrid để mục này ngoài top 20 (dòng cuối). Hybrid không thắng ở mọi câu, nhưng tính trên cả golden set nó tốt nhất (xem D.1).

In [8]:
from qdrant_client import models

nvda_10q = models.Filter(must=[
    models.FieldCondition(key="ticker", match=models.MatchValue(value="NVDA")),
    models.FieldCondition(key="filing_type", match=models.MatchValue(value="10-Q")),
])
target = "Part II — Item 1. Legal Proceedings"

def rank_of_target(query, mode):
    hits = store.search(query, k=20, query_filter=nvda_10q, mode=mode)
    ranks = [rank for rank, hit in enumerate(hits, start=1) if hit.metadata["section"] == target]
    return ranks[0] if ranks else "ngoài top 20"

query = "What is covered in Part II Item 1 of NVIDIA's 10-Q?"
for mode in ("dense", "sparse", "hybrid"):
    print(f"{mode:<7} hạng của mục đúng: {rank_of_target(query, mode)}")
print(f"\nCâu ngắn hơn, hybrid: {rank_of_target('What is covered in Part II Item 1?', 'hybrid')}")

dense   hạng của mục đúng: ngoài top 20
sparse  hạng của mục đúng: 3


hybrid  hạng của mục đúng: 4

Câu ngắn hơn, hybrid: ngoài top 20


## B.3 — `create_retriever(store, ...) -> Retriever` và `Retriever.retrieve(question, filters=None) -> list[Document]`

1. **Input:** câu hỏi; `filters` là dict (`ticker`, `filing_type`, `fiscal_year`, `fiscal_period`, `item_code`, `part`, `content_type`), mọi điều kiện phải khớp.
2. **Output:** tối đa 5 **chunk cha**, mỗi cái có `matched_chunk_ids` (các chunk con đã khớp).
3. **Chức năng:** tìm 20 chunk con tốt nhất (hybrid), (tuỳ chọn) rerank bằng cross-encoder — **tắt mặc định** vì trên CPU tốn khoảng 12 s/lần mà chỉ tăng MRR 0,014 — rồi gom theo chunk cha theo thứ tự hạng. "Tìm nhỏ, đọc lớn": chunk con ngắn cho vector sắc nét, chunk cha đủ ngữ cảnh cho LLM.
4. **Tech stack:** `FilingStore.search`, `ParentStore.get`.

In [9]:
from src.agent.state import passage_id
from src.retrieval.retriever import create_retriever

retriever = create_retriever(store)
for doc in retriever.retrieve("data center revenue growth drivers", {"ticker": "NVDA", "filing_type": "10-K"}):
    print(f"{passage_id(doc):<26} {doc.metadata['section'][:40]:<40} > {(doc.metadata.get('subsection') or '')[:45]:<45}"
          f" matched={doc.metadata.get('matched_chunk_ids')} ({len(doc.page_content):,} ký tự)")

NVDA_10-K_2026-02-25#84    Item 7. Management's Discussion and Anal > Operating Income by Reportable Segments       matched=[364, 365] (1,665 ký tự)
NVDA_10-K_2026-02-25#74    Item 7. Management's Discussion and Anal > Recent Developments, Future Objectives and Ch matched=[331, 332] (6,074 ký tự)
NVDA_10-K_2026-02-25#75    Item 7. Management's Discussion and Anal > Fiscal Year 2026 Summary                      matched=[343] (2,104 ký tự)
NVDA_10-K_2026-02-25#40    Item 1A. Risk Factors                    > Business disruptions could harm our operation matched=[189, 188, 183] (5,119 ký tự)
NVDA_10-K_2026-02-25#169   Item 15. Exhibits and Financial Statemen > Note 15 - Employee Retirement Plans > Note 16 matched=[593] (6,106 ký tự)


# PHẦN C — Agent (LangGraph)

**Các bên tham gia:** tools (`src/agent/tools.py`), chuỗi model (`src/generation/llm.py`), các node `planner`, `executor`/`run_tools`, `synthesizer` và `memory` (mỗi thứ một file trong `src/agent/`), `sidekick.py` chỉ nối chúng thành graph.

## C.1 — `create_filing_tools(store, retriever) -> list[BaseTool]`

1. **Input:** store và retriever dùng chung.
2. **Output:** 4 tool LangChain: `search_filings` (hybrid search + pre-filter), `list_available_filings` (danh mục), `list_sec_filings` (lịch sử filing trên EDGAR), `index_filing` (index một filing; metadata `requires_approval`).
3. **Chức năng:** mỗi tool trả `(content, artifact)`: text cho LLM đọc, dữ liệu có cấu trúc (Document, dict) cho code dùng. Lỗi dự kiến (ticker chưa index, không có kết quả) trả về dưới dạng thông báo để agent tự sửa.
4. **Tech stack:** `StructuredTool` + schema Pydantic (mô tả từng tham số chính là hướng dẫn cho LLM).

In [10]:
from src.agent.tools import create_filing_tools

tools = {tool.name: tool for tool in create_filing_tools(store, retriever)}
print("Tools:", list(tools))
print("Cần người dùng duyệt:", [name for name, tool in tools.items() if (tool.metadata or {}).get("requires_approval")])
message = tools["list_available_filings"].invoke({"type": "tool_call", "id": "demo", "name": "list_available_filings", "args": {}})
print("\n" + message.content)
print("\nTham số của search_filings:", list(tools["search_filings"].args_schema.model_json_schema()["properties"]))

Tools: ['search_filings', 'list_available_filings', 'list_sec_filings', 'index_filing']
Cần người dùng duyệt: ['index_filing']

Indexed filings:
- GOOGL_10-K_2026-02-05: Alphabet Inc. 10-K 2026, 10-K, fiscal 2025 FY (period ended 2025-12-31), filed 2026-02-05
- MSFT_10-K_2026-07-29: MICROSOFT CORP 10-K 2026, 10-K, fiscal 2026 FY (period ended 2026-06-30), filed 2026-07-29
- NVDA_10-Q_2026-05-20: NVIDIA CORP 10-Q 2026, 10-Q, fiscal 2027 Q1 (period ended 2026-04-26), filed 2026-05-20
- NVDA_10-K_2026-02-25: NVIDIA CORP 10-K 2026, 10-K, fiscal 2026 FY (period ended 2026-01-25), filed 2026-02-25

Tham số của search_filings: ['query', 'ticker', 'filing_type', 'fiscal_year', 'fiscal_period', 'item_code', 'part', 'content_type']


## C.2 — `get_llm(role="main" | "lite") -> ChatModel`

1. **Input:** vai trò — `main` (planner, synthesizer) hoặc `lite` (executor).
2. **Output:** một **chuỗi** model Gemini bọc bằng `with_fallbacks`: model đầu phục vụ, model sau nhận lời gọi khi model trước bị rate limit, hết quota ngày, lỗi server, timeout hay mất kết nối. Quota free của Gemini tính **theo từng model**, nên mỗi model trong chuỗi thêm một phần quota.
3. **Chức năng:** nơi duy nhất dựng LLM. Model Gemini 3 ở vai trò main chạy `thinking_level="low"` (3.8-flash: khoảng 34 s/lời gọi ở mặc định, khoảng 9 s ở "low"). Wrapper tự áp `bind_tools`/`with_structured_output` cho mọi model trong chuỗi.
4. **Tech stack:** `langchain-google-genai` (`ChatGoogleGenerativeAI`), `RunnableWithFallbacks` của `langchain-core`.

In [11]:
from src.generation.llm import get_llm

main_llm, lite_llm = get_llm("main"), get_llm("lite")
quiet_logs()
for role, chain in (("main", main_llm), ("lite", lite_llm)):
    print(f"{role:<5}", " -> ".join(model.model.removeprefix("models/") for model in [chain.runnable, *chain.fallbacks]))

2026-10-11 03:36:30 | INFO     | src.generation.llm | Gemini main chain: gemini-3.8-flash -> gemini-3.7-flash -> gemini-3.6-flash -> gemini-3.5-flash-lite


2026-10-11 03:36:32 | INFO     | src.generation.llm | Gemini lite chain: gemini-3.5-flash-lite -> gemini-3.1-flash-lite


main  gemini-3.8-flash -> gemini-3.7-flash -> gemini-3.6-flash -> gemini-3.5-flash-lite
lite  gemini-3.5-flash-lite -> gemini-3.1-flash-lite


## C.3 — `build_sidekick(store, retriever, main_llm, lite_llm) -> CompiledStateGraph`

1. **Input:** store, retriever, hai chuỗi model; tuỳ chọn `checkpointer` (mặc định trong bộ nhớ) và `max_steps` (4).
2. **Output:** graph đã compile.
3. **Chức năng:**
   - `start_turn`: mỗi lần chạy là một lượt hội thoại; xoá dữ liệu làm việc của lượt trước (bằng `Overwrite`), giữ hội thoại.
   - `planner` (main): lập kế hoạch ít bước nhất — câu đơn giản chỉ 1 bước — và **xem lại kết quả sau mỗi bước**: xong, hay còn bước nào (đổi chiến thuật nếu thiếu, ví dụ bỏ filter Item). Với câu hỏi nối tiếp, nó viết lại thành câu hỏi đầy đủ.
   - `executor` (lite) chọn tool cho bước hiện tại; `run_tools` chạy tool, dừng bằng `interrupt()` chờ duyệt trước `index_filing`.
   - `synthesizer` (main): trả lời theo ngôn ngữ người hỏi, chỉ từ các đoạn trích, citation `[n]` dựng bằng code; `end_turn` ghi câu hỏi và câu trả lời vào hội thoại.
4. **Tech stack:** `StateGraph` của LangGraph; state là `TypedDict` dữ liệu thuần (reducer cho `evidence`, `step_results`, `trace`, `messages`); `InMemorySaver`.

In [12]:
from src.agent.sidekick import build_sidekick, new_thread

agent = build_sidekick(store, retriever, main_llm, lite_llm)
quiet_logs()
print(agent.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	start_turn(start_turn)
	planner(planner)
	executor(executor)
	run_tools(run_tools)
	synthesizer(synthesizer)
	end_turn(end_turn)
	__end__([<p>__end__</p>]):::last
	__start__ --> start_turn;
	executor --> run_tools;
	planner -.-> executor;
	planner -.-> synthesizer;
	run_tools -.-> planner;
	run_tools -.-> synthesizer;
	start_turn --> planner;
	synthesizer --> end_turn;
	end_turn --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## C.4 — Chạy một câu hỏi: `agent.stream({"question": ...}, config, stream_mode="custom")`

1. **Input:** `{"question": str, "filters": dict | None}` và config chứa `thread_id` (`new_thread()`), dùng lại cho các lượt sau của cùng hội thoại.
2. **Output:** các sự kiện tiến độ trong lúc chạy; state cuối có `answer`, `citations`, `evidence`, `trace`.
3. **Chức năng:** UI stream đúng như thế này: `stream_mode="custom"` cho tiến độ, `"messages"` cho từng token của câu trả lời (chỉ lấy từ node `synthesizer`).
4. **Tech stack:** `get_stream_writer` của LangGraph (`src/agent/progress.py`).

In [13]:
from langgraph.types import Command

def run(payload, config):
    # Chạy (hoặc resume) một lượt, in tiến độ; trả về snapshot cuối.
    start = time.perf_counter()
    for event in agent.stream(payload, config, stream_mode="custom"):
        print(f"  [{time.perf_counter() - start:5.1f}s] {event['message']}")
    return agent.get_state(config)

def show(values):
    print("\nTRACE")
    for event in values["trace"]:
        print(f"  - {event['node']:<11} [{event['model']}] {event['summary'][:180]}")
    print("\nANSWER\n" + values["answer"])
    print("\nCITATIONS")
    for citation in values["citations"]:
        print(f"  [{citation['number']}] {citation['source']} — {citation['section'][:55]} › {(citation['subsection'] or '')[:50]}")

conversation = new_thread()
snapshot = run({"question": "What drove NVIDIA's data center revenue growth in fiscal 2026?"}, conversation)
show(snapshot.values)

  [  0.0s] Planning


  [  2.6s] Next: Search NVIDIA fiscal 2026 10-K (NVDA_10-K_2026-02-25) MD&A for data center revenue growth drivers.
  [  2.6s] Step 1: Search NVIDIA fiscal 2026 10-K (NVDA_10-K_2026-02-25) MD&A for data center revenue growth drivers.


  [  3.8s] Running search_filings(filing_type='10-K', fiscal_period='FY', item_code='7', ticker='NVDA', content_type='text', query='data center revenue growth drivers Compute and Networking Hopper Blackwell', fiscal_year=2026)


  [  4.1s] Reviewing step 1


  [  6.0s] Enough to answer
  [  6.0s] Writing the answer



TRACE
  - planner     [gemini-3.5-flash-lite] Initial plan | next: Search NVIDIA fiscal 2026 10-K (NVDA_10-K_2026-02-25) MD&A for data center revenue growth drivers.
  - executor    [gemini-3.5-flash-lite] search_filings(filing_type='10-K', fiscal_period='FY', item_code='7', ticker='NVDA', content_type='text', query='data center revenue growth drivers Compute and Networking Hopper Bl
  - run_tools   [None] search_filings(filing_type='10-K', fiscal_period='FY', item_code='7', ticker='NVDA', content_type='text', query='data center revenue growth drivers Compute and Networking Hopper Bl
  - planner     [gemini-3.5-flash-lite] Step 1 successfully found that NVIDIA's Data Center revenue growth in fiscal 2026 was driven by major platform shifts in accelerated computing and AI, specifically growth in Data C
  - synthesizer [gemini-3.5-flash-lite] answered from 5 passages

ANSWER
NVIDIA's Data Center revenue growth in fiscal year 2026 was driven by:

* **Major platform shifts:** Accelerated c

## C.5 — Chat nhiều lượt: câu hỏi nối tiếp trên cùng thread

1. **Input:** câu hỏi mới với **cùng** config (`thread_id`).
2. **Output:** như C.4; `standalone_question` là cách planner hiểu câu hỏi; `messages` tích luỹ cả hội thoại.
3. **Chức năng:** ở lời gọi đầu của lượt, planner đọc 3 lượt gần nhất và viết lại câu nối tiếp ("còn Microsoft?") thành câu đầy đủ, không tốn thêm lời gọi LLM nào. Executor và synthesizer làm việc trên câu đã viết lại; câu trả lời vẫn theo ngôn ngữ người hỏi.
4. **Tech stack:** checkpointer của LangGraph giữ state theo thread; reducer `add_messages`; `Overwrite` để reset dữ liệu theo lượt (`src/agent/memory.py`).

In [14]:
snapshot = run({"question": "Còn Microsoft thì sao? Mảng cloud của họ tăng trưởng nhờ đâu?"}, conversation)
print("\nPlanner hiểu câu hỏi là:", snapshot.values["standalone_question"])
show(snapshot.values)
print(f"\nHội thoại hiện có {len(snapshot.values['messages'])} message (2 lượt × hỏi/đáp)")

  [  0.1s] Planning


  [  2.0s] Next: Search Microsoft fiscal 2026 10-K MD&A for cloud and Azure revenue growth drivers
  [  2.0s] Step 1: Search Microsoft fiscal 2026 10-K MD&A for cloud and Azure revenue growth drivers


  [  2.8s] Running search_filings(item_code='7', fiscal_period='FY', ticker='MSFT', filing_type='10-K', query='cloud revenue Azure growth drivers Intelligence Cloud', fiscal_year=2026)


  [  3.0s] Reviewing step 1


  [  5.2s] Enough to answer
  [  5.2s] Writing the answer



Planner hiểu câu hỏi là: What drove Microsoft's cloud revenue growth in its latest fiscal year 2026?

TRACE
  - planner     [gemini-3.5-flash-lite] Read as: What drove Microsoft's cloud revenue growth in its latest fiscal year 2026? | Initial plan | next: Search Microsoft fiscal 2026 10-K MD&A for cloud and Azure revenue growt
  - executor    [gemini-3.5-flash-lite] search_filings(item_code='7', fiscal_period='FY', ticker='MSFT', filing_type='10-K', query='cloud revenue Azure growth drivers Intelligence Cloud', fiscal_year=2026)
  - run_tools   [None] search_filings(item_code='7', fiscal_period='FY', ticker='MSFT', filing_type='10-K', query='cloud revenue Azure growth drivers Intelligence Cloud', fiscal_year=2026) -> 5 passages
  - planner     [gemini-3.5-flash-lite] The first step successfully retrieved details about Microsoft's Intelligent Cloud and Azure growth drivers in fiscal year 2026, showing Azure and other cloud services revenue grew 
  - synthesizer [gemini-3.5-flash-lite] 

## C.6 — Duyệt trước khi index: `interrupt()` và `Command(resume=...)`

1. **Input:** một câu hỏi về công ty chưa được index.
2. **Output:** graph **dừng** ở `run_tools`; `snapshot.interrupts[0].value` mô tả yêu cầu (tool, tham số, bước). Resume bằng `Command(resume={"approved": bool})`.
3. **Chức năng:** planner lên kế hoạch `list_sec_filings` → `index_filing` (ghi rõ ngày nộp) → tìm kiếm. Mọi yêu cầu duyệt được hỏi **trước khi** chạy tool nào, và tách khỏi lời gọi LLM của executor, nên khi resume không có gì chạy lại hai lần. Bị từ chối thì kết quả được ghi lại, planner dừng và câu trả lời giải thích lý do.
4. **Tech stack:** `interrupt` / `Command` của LangGraph (cần checkpointer).

Ở đây yêu cầu bị **từ chối**, nên store không đổi.

In [15]:
other = new_thread()
snapshot = run({"question": "What were Apple's total net sales in its most recent 10-K?"}, other)
request = snapshot.interrupts[0].value
print("\nYÊU CẦU DUYỆT:", request["tool"], request["args"], "| bước:", request["step"])

snapshot = run(Command(resume={"approved": False}), other)
show(snapshot.values)

  [  0.0s] Planning


  [  1.9s] Next: List Apple SEC filings to find its most recent 10-K.
  [  1.9s] Step 1: List Apple SEC filings to find its most recent 10-K.


  [  2.9s] Running list_sec_filings(ticker='AAPL', limit=5, filing_type='10-K')


  [  4.9s] Reviewing step 1


  [  7.0s] Next: Index Apple's 10-K filed on 2025-10-31.
  [  7.0s] Step 2: Index Apple's 10-K filed on 2025-10-31.



YÊU CẦU DUYỆT: index_filing {'filing_date': '2025-10-31', 'filing_type': '10-K', 'ticker': 'AAPL'} | bước: Index Apple's 10-K filed on 2025-10-31.
  [  0.0s] Skipped index_filing: declined
  [  0.0s] Reviewing step 2


  [  1.9s] Enough to answer
  [  1.9s] Writing the answer



TRACE
  - planner     [gemini-3.5-flash-lite] Initial plan | next: List Apple SEC filings to find its most recent 10-K.
  - executor    [gemini-3.5-flash-lite] list_sec_filings(ticker='AAPL', limit=5, filing_type='10-K')
  - run_tools   [None] list_sec_filings(ticker='AAPL', limit=5, filing_type='10-K') -> Apple Inc. (AAPL) 10-K filings on SEC EDGAR, newest first:
  - planner     [gemini-3.5-flash-lite] Step 1 successfully found Apple's most recent 10-K filed on 2025-10-31 for the period ended 2025-09-27. Since this filing is not indexed yet, we need to index it before we can sear
  - executor    [gemini-3.5-flash-lite] index_filing(filing_date='2025-10-31', filing_type='10-K', ticker='AAPL')
  - run_tools   [None] index_filing(filing_date='2025-10-31', filing_type='10-K', ticker='AAPL') -> Not run: the user declined.
  - planner     [gemini-3.5-flash-lite] The user declined to index Apple's 10-K filing, so we cannot retrieve Apple's total net sales. | done
  - synthesizer [gemini-3.5

# PHẦN D — Đánh giá

**Các bên tham gia:** `src/evaluation/metrics.py` (retrieval, không gọi LLM) và `src/evaluation/agent_eval.py` (agent đầu-cuối, tốn quota LLM).

## D.1 — `load_golden_set()` + `evaluate_retrieval(retriever, cases, k=5) -> dict`

1. **Input:** retriever; golden set 35 câu, mỗi câu có các mục liên quan ở mức `(doc_id, section)` (bền qua mọi lần chunk lại); `k`.
2. **Output:** `{"per_case": [...], "aggregate": {"hit_rate@k", "mrr@k", "recall@k", "precision@k"}}`.
3. **Chức năng:** đo riêng chất lượng truy xuất, một lần tìm cho mỗi câu, không có LLM. CLI: `python -m src.evaluation.metrics --save <label> --compare <baseline>`.
4. **Tech stack:** Python thuần; kết quả lưu ở `data/eval/<label>.json`.

In [16]:
from src.evaluation.metrics import evaluate_retrieval, load_golden_set

quiet_logs()
cases = load_golden_set()
start = time.perf_counter()
retrieval = evaluate_retrieval(retriever, cases, k=5)
print(f"{len(cases)} câu trong {time.perf_counter() - start:.1f}s")
{metric: round(value, 3) for metric, value in retrieval["aggregate"].items()}

35 câu trong 6.0s


{'hit_rate@k': 0.943, 'mrr@k': 0.86, 'recall@k': 0.914, 'precision@k': 0.457}

## D.2 — `load_agent_cases()` + `evaluate_agent(graph, cases, new_thread) -> dict`

1. **Input:** graph agent; bộ câu hỏi **nhiều bước** (`agent_golden_set.json`): so sánh hai/ba công ty, một con số kèm động lực tăng trưởng, hai filing của cùng công ty, một lượt hỏi nối tiếp.
2. **Output:** cho từng câu — `hop_recall` (tỷ lệ "hop" tìm được: mỗi hop là nơi chứa một dữ kiện, có thể có nhiều mục thay thế), `term_recall` (tỷ lệ con số then chốt có trong câu trả lời, chấp nhận nhiều cách viết), số bước, số lời gọi LLM, thời gian, model đã phục vụ.
3. **Chức năng:** đo cả agent: kế hoạch, tool, review, câu trả lời. Mỗi câu chạy trên thread mới; yêu cầu index bị từ chối tự động. CLI: `python -m src.evaluation.agent_eval --save <label>`.
4. **Tech stack:** chính graph ở Phần C.

Chạy lại tốn khoảng 40 lời gọi LLM, nên ở đây chỉ đọc kết quả đã lưu của lần chạy Phase 6:

In [17]:
from src.evaluation.agent_eval import load_agent_cases

agent_cases = load_agent_cases()
example = next(case for case in agent_cases if case.id == "revenue-msft-googl")
print("Ví dụ:", example.turns, "\n  hops:", [sorted(hop) for hop in example.hops], "\n  answer_terms:", example.answer_terms)

saved = json.loads((ROOT / "data" / "eval" / "agent_phase6.json").read_text(encoding="utf-8"))
print("\nLần chạy:", saved["timestamp"][:19], "| chuỗi main:", " -> ".join(saved["config"]["main_chain"]))
display(pd.DataFrame(saved["per_case"])[["id", "kind", "hop_recall", "term_recall", "steps", "llm_calls", "seconds"]])
saved["aggregate"]

Ví dụ: ["Compare Microsoft's and Alphabet's total revenue for their most recent fiscal years."] 
  hops: [[('MSFT_10-K_2026-07-29', 'ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATIONS'), ('MSFT_10-K_2026-07-29', 'ITEM 8. FINANCIAL STATEMENTS AND SUPPLEMENTARY DATA')], [('GOOGL_10-K_2026-02-05', 'ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS OF FINANCIAL CONDITION AND RESULTS OF OPERATIONS'), ('GOOGL_10-K_2026-02-05', 'ITEM 8. FINANCIAL STATEMENTS AND SUPPLEMENTARY DATA')]] 
  answer_terms: [['331,839', '331.8 billion', '331.8'], ['402,836', '402.8 billion', '402.8']]

Lần chạy: 2026-10-10T20:24:58 | chuỗi main: gemini-3.8-flash -> gemini-3.7-flash -> gemini-3.6-flash -> gemini-3.5-flash-lite


,id,kind,hop_recall,term_recall,steps,llm_calls,seconds
0,nvda-dc-revenue-vi,"figure + drivers, Vietnamese",1.0,1.0,3,8,19.3
1,employees-nvda-msft,comparison across companies,1.0,1.0,2,6,11.5
2,revenue-msft-googl,"comparison across companies, figures",1.0,1.0,2,6,12.3
3,cyber-governance-3way,"three companies, one section each",1.0,NaN,3,8,51.2
4,nvda-quarter-vs-year,two filings of one company,1.0,1.0,3,8,15.6
5,follow-up-employees-vi,"follow-up turn, Vietnamese",1.0,1.0,1,4,49.5


{'cases': 6,
 'failed': 0,
 'hop_recall': 1.0,
 'all_hops_rate': 1.0,
 'term_recall': 1.0,
 'mean_steps': 2.333,
 'mean_seconds': 26.567,
 'llm_calls': 40,
 'models': {'gemini-3.5-flash-lite': 40}}

In [18]:
store.close()  # nhả khoá thư mục data/qdrant cho app và CLI

# Tổng kết

| Luồng | Module | Điểm chính |
|---|---|---|
| A. Index | `filing_facts`, `parse_html`, `chunking`, `embedder`, `vector_store`, `parent_store`, `pipeline` | Kỳ tài chính đọc từ XBRL; H2 = Item, H3 = tiêu đề phụ; chunk cha (đọc) / chunk con (tìm); mỗi chunk con 2 vector |
| B. Truy vấn | `vector_store.search`, `retriever` | Hybrid dense + BM25 gộp bằng RRF trong Qdrant; filter áp trong lúc tìm; trả về chunk cha |
| C. Agent | `src/agent/` (`memory`, `planner`, `executor`, `synthesizer`, `tools`, `sidekick`), `generation/llm.py` | Planner ⇄ executor có review sau mỗi bước; chat nhiều lượt; duyệt trước khi index; chuỗi fallback model Gemini |
| D. Đánh giá | `evaluation/metrics.py`, `evaluation/agent_eval.py` | Retrieval: hit rate / MRR @5; agent: hop recall, term recall, bước, thời gian |

Các quyết định thiết kế và số liệu đo được ghi trong `docs/DESIGN_DECISIONS.md`; cách chạy app và CLI trong `README.md`.